In [110]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [111]:
PROJECT_DIR = Path.cwd().resolve().parent
DATA_DIR = PROJECT_DIR / 'data'
SRC_DIR = PROJECT_DIR / 'src'
EXTRACT_DIR = PROJECT_DIR / 'processed' / 'extraction'
OUTPUT_DIR = PROJECT_DIR / 'processed' / 'transform'
REPORT_DIR = PROJECT_DIR / 'reports'

print('PROJECT_DIR:', PROJECT_DIR)
print('EXTRACT_DIR:', EXTRACT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

PROJECT_DIR: /Users/oneash/ONEASH_local/Coding/Delirium-Prediction/Parkinson
EXTRACT_DIR: /Users/oneash/ONEASH_local/Coding/Delirium-Prediction/Parkinson/processed/extraction
OUTPUT_DIR: /Users/oneash/ONEASH_local/Coding/Delirium-Prediction/Parkinson/processed/transform


## 데이터 로드

In [112]:
# all events (chart + lab + eMAR medication point events)
all_events = pd.read_csv(EXTRACT_DIR / 'all_events_long.csv')

print(f"all_events shape: {all_events.shape}")
print(f"Columns: {all_events.columns.tolist()}")
all_events.head()


all_events shape: (816682, 12)
Columns: ['source_table', 'subject_id', 'hadm_id', 'stay_id', 'charttime', 'itemid', 'label', 'feature_name', 'type', 'value', 'valuenum', 'valueuom']


,source_table,subject_id,hadm_id,stay_id,charttime,itemid,label,feature_name,type,value,valuenum,valueuom
0,chartevents,14979348,22895491.0,39414466,2139-08-31 16:00:00,225664.0,Glucose finger stick (range 70-100),glucose,numeric,87,87.0,NaN
1,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220179.0,Non Invasive Blood Pressure systolic,nibp_sbp,numeric,170,170.0,mmHg
2,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220180.0,Non Invasive Blood Pressure diastolic,nibp_dbp,numeric,80,80.0,mmHg
3,chartevents,14979348,22895491.0,39414466,2139-08-30 23:50:00,220181.0,Non Invasive Blood Pressure mean,nibp_mbp,numeric,102,102.0,mmHg
4,chartevents,14979348,22895491.0,39414466,2139-08-30 23:51:00,220045.0,Heart Rate,heart_rate,numeric,93,93.0,bpm


In [113]:
# adm_pat_icu (icu stay info)
adm_pat_icu = pd.read_csv(EXTRACT_DIR / 'adm_pat_icu.csv')
print(f"adm_pat_icu shape: {adm_pat_icu.shape}")
print(f"Columns: {adm_pat_icu.columns.tolist()}")
adm_pat_icu.head()

adm_pat_icu shape: (974, 19)
Columns: ['subject_id', 'hadm_id', 'stay_id', 'first_careunit', 'last_careunit', 'intime', 'outtime', 'los', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group', 'dod', 'admittime', 'dischtime', 'deathtime', 'admission_type', 'race', 'icu_los_hours']


,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,gender,anchor_age,anchor_year,anchor_year_group,dod,admittime,dischtime,deathtime,admission_type,race,icu_los_hours
0,10018328,23786647,31269608,Neuro Stepdown,Neuro Stepdown,2154-04-24 23:03:44,2154-05-02 15:55:21,7.702512,F,83,2154,2014 - 2016,2158-12-14,2154-04-24 03:15:00,2154-05-03 14:00:00,NaN,SURGICAL SAME DAY ADMISSION,WHITE,184.860278
1,10018328,28252562,37006782,Neuro Intermediate,Neuro Intermediate,2158-02-08 22:56:08,2158-02-13 23:58:23,5.043229,F,83,2154,2014 - 2016,2158-12-14,2158-02-08 22:55:00,2158-02-18 17:30:00,NaN,OBSERVATION ADMIT,WHITE,121.037500
2,10050755,20050796,37743005,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2134-02-26 01:03:02,2134-02-26 04:27:45,0.142164,M,77,2126,2008 - 2010,2134-02-26,2134-02-25 18:41:00,2134-02-26 01:26:00,2134-02-26 01:26:00,OBSERVATION ADMIT,WHITE - RUSSIAN,3.411944
3,10052769,22087051,30336368,Neuro Surgical Intensive Care Unit (Neuro SICU),Neuro Surgical Intensive Care Unit (Neuro SICU),2124-06-16 20:18:49,2124-07-08 14:31:39,21.758912,M,78,2124,2020 - 2022,2124-07-09,2124-04-25 19:33:00,2124-07-09 09:43:00,2124-07-09 09:43:00,URGENT,UNKNOWN,522.213889
4,10052769,22087051,34171709,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2124-05-25 15:48:47,2124-05-26 02:14:53,0.434792,M,78,2124,2020 - 2022,2124-07-09,2124-04-25 19:33:00,2124-07-09 09:43:00,2124-07-09 09:43:00,URGENT,UNKNOWN,10.435000


In [114]:
# Date columns
all_events['charttime'] = pd.to_datetime(all_events['charttime'], errors='coerce')
for col in ['intime', 'outtime', 'admittime', 'dischtime', 'deathtime']:
    adm_pat_icu[col] = pd.to_datetime(adm_pat_icu[col], errors='coerce')

## VALUE 변환 (문자열 → 숫자)

In [115]:
all_events.rename(columns={'value': 'value_str',
                           'valuenum': 'value'}, inplace=True)

In [116]:
# 문자열/범주형 -> 숫자 변환

# Delirium assessment: Positive=1, Negative=0, UTA/other=NaN
all_events.loc[all_events['value_str'].astype('string').str.lower().eq('positive'), 'value'] = 1
all_events.loc[all_events['value_str'].astype('string').str.lower().eq('negative'), 'value'] = 0

# RASS text values
all_events.loc[all_events['value_str'] == '"+4 Combative' , 'value']        = 4
all_events.loc[all_events['value_str'] == '+3 Pulls or removes tube(s) or catheter(s); aggressive', 'value'] = 3
all_events.loc[all_events['value_str'] == '"+2 Frequent nonpurposeful movement', 'value']                    = 2
all_events.loc[all_events['value_str'] == '"+1 Anxious', 'value']           = 1
all_events.loc[all_events['value_str'] == ' 0  Alert and calm', 'value']    = 0
all_events.loc[all_events['value_str'] == '-1 Awakens to voice (eye opening/contact) > 10 sec', 'value'] = -1
all_events.loc[all_events['value_str'] == '"-2 Light sedation' , 'value']   = -2
all_events.loc[all_events['value_str'] == '"-3 Moderate sedation', 'value'] = -3
all_events.loc[all_events['value_str'] == '"-4 Deep sedation', 'value']     = -4
all_events.loc[all_events['value_str'] == '"-5 Unarousable' , 'value']      = -5


## 단위 변환

In [117]:
if 'valueuom_original' not in all_events.columns:
    all_events['valueuom_original'] = all_events['valueuom']

def fahr_to_celsius(temp_fahr):
    return (temp_fahr - 32) * 5 / 9

# Temperature: Fahrenheit -> Celsius
temp_f_mask = all_events['label'].eq('Temperature Fahrenheit')
all_events.loc[temp_f_mask, 'value'] = fahr_to_celsius(all_events.loc[temp_f_mask, 'value'])
all_events.loc[all_events['feature_name'].eq('temperature'), 'valueuom'] = 'C'

# Weight: lbs -> kg
weight_lbs_mask = all_events['label'].eq('Admission Weight (lbs.)')
all_events.loc[weight_lbs_mask, 'value'] = all_events.loc[weight_lbs_mask, 'value'] * 0.453592
all_events.loc[all_events['feature_name'].eq('weight'), 'valueuom'] = 'kg'

# Height: inch -> cm
height_inch_mask = all_events['label'].eq('Height')
all_events.loc[height_inch_mask, 'value'] = all_events.loc[height_inch_mask, 'value'] * 2.54
all_events.loc[all_events['feature_name'].eq('height'), 'valueuom'] = 'cm'


In [118]:
all_events.to_csv(OUTPUT_DIR / 'all_events_filtered.csv', index=False)


## 시간 계산 (ICU 입실 기준)

In [119]:
# length of stay in icu (hours)
adm_pat_icu['los_hours'] = ((adm_pat_icu['outtime'] - adm_pat_icu['intime']).dt.total_seconds() / 3600).astype(int)

In [120]:
patient_cols = [
    'stay_id', 'subject_id', 'hadm_id', 'anchor_age', 'gender', 'los_hours',
    'admission_type', 'race', 'hospital_expire_flag', 'intime', 'outtime'
]
patient_cols = [c for c in patient_cols if c in adm_pat_icu.columns]
merge_keys = [c for c in ['stay_id', 'subject_id', 'hadm_id'] if c in all_events.columns and c in patient_cols]
all_events = all_events.merge(adm_pat_icu[patient_cols], on=merge_keys, how='inner')

# ICU 입실 후 경과시간 계산
all_events['hours'] = ((all_events['charttime'] - all_events['intime']).dt.total_seconds() / 3600.0)

# Keep events within ICU stay
all_events = all_events[(all_events['hours'] >= 0) & (all_events['charttime'] <= all_events['outtime'])].copy()

# 60-minute binning
all_events['bin'] = all_events['hours'].astype(int)

# 약물은 all_events_long에 point event로 포함되어 있으므로 실제 투약이 기록된 hour만 1로 pivot됩니다.
medication_feature_cols = sorted(all_events.loc[all_events['source_table'].eq('emar'), 'feature_name'].dropna().unique().tolist())
print(f"Medication point-event features: {medication_feature_cols}")


Medication point-event features: ['amantadine', 'anticholinergic', 'antipsychotic', 'benzodiazepine', 'comt_inhibitor', 'dopamine_agonist', 'levodopa_related', 'maob_inhibitor', 'opioid', 'sedatives', 'vasopressor']


## 60분 비닝 및 피봇 (최종 시계열)

In [121]:
# 시간 bin별 기본 정보 (나이, 성별, LOS, 입원 유형 등 시간별 행에 반복)
base_agg = {
    'hours': 'max',
    'anchor_age': 'max',
    'gender': 'first',
    'los_hours': 'max',
    'subject_id': 'first',
    'hadm_id': 'first',
    'race': 'first',
}
base_agg = {col: agg for col, agg in base_agg.items() if col in all_events.columns}
base_info = all_events.groupby(['stay_id', 'bin']).agg(base_agg).reset_index()
base_info.rename(columns={'anchor_age': 'age'}, inplace=True)
base_info.head(10)

,stay_id,bin,hours,age,gender,los_hours,subject_id,hadm_id,race
0,30008046,0,0.669722,75,M,99,19448158,28451027.0,WHITE
1,30008046,1,1.686389,75,M,99,19448158,28451027.0,WHITE
2,30008046,2,2.403056,75,M,99,19448158,28451027.0,WHITE
3,30008046,4,4.403056,75,M,99,19448158,28451027.0,WHITE
4,30008046,6,6.403056,75,M,99,19448158,28451027.0,WHITE
5,30008046,8,8.403056,75,M,99,19448158,28451027.0,WHITE
6,30008046,10,10.403056,75,M,99,19448158,28451027.0,WHITE
7,30008046,11,11.336389,75,M,99,19448158,28451027.0,WHITE
8,30008046,12,12.403056,75,M,99,19448158,28451027.0,WHITE
9,30008046,14,14.403056,75,M,99,19448158,28451027.0,WHITE


In [122]:
# 이벤트 long-format 데이터를 stay_id-bin 단위 wide-format으로 변환
# 같은 시간 bin에 여러 값이 있으면 max 값 사용
pivoted = all_events.pivot_table(
    index=['stay_id', 'bin'],
    columns='feature_name',
    values='value',
    aggfunc='max'
).reset_index()
pivoted.columns.name = None
pivoted.head(10)

,stay_id,bin,BUN,abp_dbp,abp_mbp,abp_sbp,albumin,alt,amantadine,anticholinergic,...,rass,respiratory_rate,sao2,sedatives,sodium,spo2,temperature,vasopressor,wbc,weight
0,30008046,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,29.0,NaN,NaN,NaN,97.0,37.111111,NaN,NaN,NaN
1,30008046,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.147609
2,30008046,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,25.0,NaN,NaN,NaN,97.0,36.666667,NaN,NaN,NaN
3,30008046,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,23.0,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
4,30008046,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,19.0,NaN,NaN,NaN,99.0,36.444444,NaN,NaN,NaN
5,30008046,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,19.0,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
6,30008046,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,22.0,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
7,30008046,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,36.333333,NaN,NaN,NaN
8,30008046,12,23.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,22.0,NaN,NaN,139.0,100.0,NaN,NaN,5.3,NaN
9,30008046,14,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.0,20.0,NaN,NaN,NaN,99.0,36.666667,NaN,NaN,NaN


In [123]:
# 기본 정보와 pivot 결과를 결합
timeseries = base_info.merge(pivoted, on=['stay_id', 'bin'], how='left')
timeseries = timeseries.sort_values(['stay_id', 'bin']).reset_index(drop=True)

print("\n=== Final Timeseries ===")
print(f"Unique stay_ids: {timeseries['stay_id'].nunique()}")
print(f"Total rows: {len(timeseries)}")
print(f"\nColumns: {timeseries.columns.tolist()}")
display(timeseries.head(10))
timeseries.to_csv(OUTPUT_DIR / 'all_events_timeseries.csv', index=False)


=== Final Timeseries ===
Unique stay_ids: 974
Total rows: 86648

Columns: ['stay_id', 'bin', 'hours', 'age', 'gender', 'los_hours', 'subject_id', 'hadm_id', 'race', 'BUN', 'abp_dbp', 'abp_mbp', 'abp_sbp', 'albumin', 'alt', 'amantadine', 'anticholinergic', 'antipsychotic', 'ast', 'benzodiazepine', 'bicarbonate', 'bicarbonate_gas', 'bilirubin_direct', 'bilirubin_indirect', 'bilirubin_total', 'calcium_free', 'chloride', 'comt_inhibitor', 'creatinine', 'delirium', 'dopamine_agonist', 'gcs_eye', 'gcs_motor', 'gcs_verbal', 'glucose', 'heart_rate', 'height', 'hematocrit', 'hemoglobin', 'inr_pt', 'lactate', 'levodopa_related', 'magnesium', 'maob_inhibitor', 'nibp_dbp', 'nibp_mbp', 'nibp_sbp', 'opioid', 'pco2', 'ph', 'phosphate', 'platelet_count', 'po2', 'potassium', 'pt', 'ptt', 'rass', 'respiratory_rate', 'sao2', 'sedatives', 'sodium', 'spo2', 'temperature', 'vasopressor', 'wbc', 'weight']


,stay_id,bin,hours,age,gender,los_hours,subject_id,hadm_id,race,BUN,...,rass,respiratory_rate,sao2,sedatives,sodium,spo2,temperature,vasopressor,wbc,weight
0,30008046,0,0.669722,75,M,99,19448158,28451027.0,WHITE,NaN,...,0.0,29.0,NaN,NaN,NaN,97.0,37.111111,NaN,NaN,NaN
1,30008046,1,1.686389,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,81.147609
2,30008046,2,2.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,0.0,25.0,NaN,NaN,NaN,97.0,36.666667,NaN,NaN,NaN
3,30008046,4,4.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,23.0,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
4,30008046,6,6.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,19.0,NaN,NaN,NaN,99.0,36.444444,NaN,NaN,NaN
5,30008046,8,8.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,19.0,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
6,30008046,10,10.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,22.0,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN
7,30008046,11,11.336389,75,M,99,19448158,28451027.0,WHITE,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,36.333333,NaN,NaN,NaN
8,30008046,12,12.403056,75,M,99,19448158,28451027.0,WHITE,23.0,...,NaN,22.0,NaN,NaN,139.0,100.0,NaN,NaN,5.3,NaN
9,30008046,14,14.403056,75,M,99,19448158,28451027.0,WHITE,NaN,...,-1.0,20.0,NaN,NaN,NaN,99.0,36.666667,NaN,NaN,NaN


## 처치/장치 노출 병합


In [124]:
procedure_events_path = EXTRACT_DIR / 'procedure_selected.csv'
procedure_events = pd.read_csv(procedure_events_path) if procedure_events_path.exists() else pd.DataFrame()

for col in ['starttime', 'endtime', 'intime', 'outtime']:
    if len(procedure_events) and col in procedure_events.columns:
        procedure_events[col] = pd.to_datetime(procedure_events[col], errors='coerce')

In [125]:
# 처치/장치 이벤트 구간이 겹치는 모든 hourly bin을 노출로 표시
procedure_flags = pd.DataFrame(columns=['stay_id', 'bin'])

proc = procedure_events.copy()
proc['start_use'] = proc['starttime']
proc['end_use']   = proc['endtime'].fillna(proc['starttime'])
proc['start_bin'] = np.floor((proc['start_use'] - proc['intime']).dt.total_seconds() / 3600).astype('int64').clip(lower=0)
proc['end_bin']   = np.ceil((proc['end_use'] - proc['intime']).dt.total_seconds() / 3600).astype('int64')
proc['max_bin']   = np.ceil((proc['outtime'] - proc['intime']).dt.total_seconds() / 3600).astype('int64')
proc['end_bin']   = proc[['end_bin', 'max_bin']].min(axis=1)
expanded = []
for row_ in proc[['stay_id', 'feature_name', 'start_bin', 'end_bin']].itertuples(index=False):
    if pd.isna(row_.feature_name):
        continue
    start_bin = max(0, int(row_.start_bin))
    end_bin = max(start_bin, int(row_.end_bin))
    expanded.append(pd.DataFrame({
        'stay_id': row_.stay_id,
        'bin': np.arange(start_bin, end_bin + 1, dtype='int64'),
        'procedure_feature_name': row_.feature_name,
        'value': 1,
    }))
if expanded:
    procedure_expanded = pd.concat(expanded, ignore_index=True).drop_duplicates(['stay_id', 'bin', 'procedure_feature_name'])
    procedure_flags = procedure_expanded.pivot_table(
        index=['stay_id', 'bin'],
        columns='procedure_feature_name',
        values='value',
        aggfunc='max',
        fill_value=0,
    ).reset_index()
    procedure_flags.columns.name = None

print('procedure_flags:', procedure_flags.shape)

procedure_flags: (26941, 4)


In [126]:
# timeseries와 결합
timeseries = timeseries.merge(procedure_flags, on=['stay_id', 'bin'], how='left')

med_binary_cols = [c for c in medication_feature_cols if c in timeseries.columns]
procedure_binary_cols = [c for c in timeseries.columns if c.startswith('procedure_')]
binary_cols = sorted(set(med_binary_cols + procedure_binary_cols))

timeseries[binary_cols] = timeseries[binary_cols].fillna(0)
for col in binary_cols:
    timeseries[col] = (timeseries[col] > 0).astype('int8')

In [127]:
timeseries[timeseries.procedure_invasive_ventilation == 1].head()

,stay_id,bin,hours,age,gender,los_hours,subject_id,hadm_id,race,BUN,...,sao2,sedatives,sodium,spo2,temperature,vasopressor,wbc,weight,procedure_invasive_ventilation,procedure_noninvasive_ventilation
349,30054857,0,0.866667,77,M,152,10476294,28537603.0,UNKNOWN,24.0,...,NaN,0,140.0,94.0,34.1,0,5.5,72.3,1,0
350,30054857,1,1.916667,77,M,152,10476294,28537603.0,UNKNOWN,NaN,...,NaN,1,NaN,100.0,34.3,0,NaN,NaN,1,0
351,30054857,2,2.600000,77,M,152,10476294,28537603.0,UNKNOWN,NaN,...,NaN,0,NaN,100.0,NaN,0,NaN,NaN,1,0
352,30054857,3,3.616667,77,M,152,10476294,28537603.0,UNKNOWN,NaN,...,NaN,0,NaN,100.0,35.4,0,NaN,NaN,1,0
353,30054857,4,4.083333,77,M,152,10476294,28537603.0,UNKNOWN,NaN,...,NaN,0,NaN,100.0,35.7,0,NaN,NaN,1,0


## 기본정보 보간

`weight`, `height`만 같은 ICU stay 안에서 첫 non-null 측정값을 전체 시간축에 채움


In [128]:
static_fill_cols = ['weight', 'height']

def first_observed_value(values):
    observed = values.dropna()
    if len(observed) == 0:
        return np.nan
    return observed.iloc[0]

for col in static_fill_cols:
    before_null = timeseries[col].isnull().sum()
    first_values = timeseries.sort_values(['stay_id', 'bin']).groupby('stay_id')[col].transform(first_observed_value)
    timeseries[col] = first_values
    after_null = timeseries[col].isnull().sum()

timeseries[static_fill_cols].info()

<class 'pandas.DataFrame'>
RangeIndex: 86648 entries, 0 to 86647
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   weight  78448 non-null  float64
 1   height  2763 non-null   float64
dtypes: float64(2)
memory usage: 1.3 MB


In [129]:
timeseries.to_csv(OUTPUT_DIR / 'all_timeseries.csv', index=False)

## 포함/제외 기준 적용

In [130]:
def _cohort_counts(step, cohort_df, timeseries_df):
    stay_ids = set(cohort_df['stay_id'])
    subset = timeseries_df[timeseries_df['stay_id'].isin(stay_ids)]
    return {
        'step': step,
        'n_subjects': int(cohort_df['subject_id'].nunique()),
        'n_hadm': int(cohort_df['hadm_id'].nunique()),
        'n_stays': int(cohort_df['stay_id'].nunique()),
        'timeseries_rows': int(len(subset)),
        'assessment_rows': int(subset['delirium'].notna().sum()),
    }

adm_for_cohort = adm_pat_icu.copy()
for col in ['intime', 'outtime']:
    adm_for_cohort[col] = pd.to_datetime(adm_for_cohort[col], errors='coerce')
adm_for_cohort['icu_los_hours'] = (adm_for_cohort['outtime'] - adm_for_cohort['intime']).dt.total_seconds() / 3600

# All ICU stays from extraction
cohort_flow_rows = []
cohort = adm_for_cohort.copy()
cohort_flow_rows.append(_cohort_counts('0. All ICU stays from extraction', cohort, timeseries))

# Positive ICU LOS
cohort = cohort[cohort['icu_los_hours'] > 0].copy()
cohort_flow_rows.append(_cohort_counts('1. Positive ICU LOS', cohort, timeseries))

# ICU LOS >= 24 hours
cohort = cohort[cohort['icu_los_hours'] >= 24].copy()
cohort_flow_rows.append(_cohort_counts('2. ICU LOS >= 24 hours', cohort, timeseries))

# Delirium assessment available after 24 hours from ICU admission
assessment_stay_ids = set(timeseries.loc[timeseries['delirium'].notna() & (timeseries['hours'] >= 24), 'stay_id'])
cohort = cohort[cohort['stay_id'].isin(assessment_stay_ids)].copy()
cohort_flow_rows.append(_cohort_counts('3. Delirium assessment available after 24 hours', cohort, timeseries))


cohort_final = cohort.copy()
cohort_stay_ids = set(cohort_final['stay_id'])
timeseries_cohort = timeseries[timeseries['stay_id'].isin(cohort_stay_ids)].copy()

cohort_flow = pd.DataFrame(cohort_flow_rows)
cohort_flow['stays_removed_from_previous'] = cohort_flow['n_stays'].shift(1).fillna(cohort_flow['n_stays']).astype(int) - cohort_flow['n_stays'].astype(int)
cohort_flow['stays_pct_of_initial'] = cohort_flow['n_stays'] / cohort_flow.loc[0, 'n_stays'] * 100
cohort_flow.to_csv(OUTPUT_DIR / 'cohort_attrition.csv', index=False)

print(f"Cohort timeseries: {timeseries_cohort.shape}")
display(cohort_flow)


Cohort timeseries: (74230, 68)


,step,n_subjects,n_hadm,n_stays,timeseries_rows,assessment_rows,stays_removed_from_previous,stays_pct_of_initial
0,0. All ICU stays from extraction,568,850,974,86648,5854,0,100.000000
1,1. Positive ICU LOS,568,850,974,86648,5854,0,100.000000
2,2. ICU LOS >= 24 hours,489,690,775,83689,5588,199,79.568789
3,3. Delirium assessment available after 24 hours,390,514,573,74230,5320,202,58.829569


## ever_delirium 라벨 생성

subject-level label. 같은 subject에서 `delirium == 1`이 한 번이라도 있으면 1, 없으면 0


In [131]:
subject_ever_delirium = (
    timeseries_cohort
    .groupby('subject_id')['delirium']
    .max()
    .fillna(0)
    .astype('int8')
)

timeseries_cohort['ever_delirium'] = timeseries_cohort['subject_id'].map(subject_ever_delirium).astype('int8')
cohort_final['ever_delirium'] = cohort_final['subject_id'].map(subject_ever_delirium).astype('int8')

print('ever_delirium subject 분포:')
print(subject_ever_delirium.value_counts().sort_index())
print()
print('ever_delirium stay 분포:')
print(cohort_final.groupby('stay_id')['ever_delirium'].first().value_counts().sort_index())


ever_delirium subject 분포:
delirium
0    137
1    253
Name: count, dtype: int64

ever_delirium stay 분포:
ever_delirium
0    160
1    413
Name: count, dtype: int64


## 산출물 저장


In [132]:
# 다음 단계에서 사용할 최소 산출물을 저장합니다.
assessment_index = (
    timeseries_cohort
    .loc[timeseries_cohort['delirium'].notna(), ['subject_id', 'stay_id', 'bin', 'delirium', 'ever_delirium']]
    .rename(columns={'bin': 'assessment_bin'})
    .sort_values(['subject_id', 'stay_id', 'assessment_bin'])
    .reset_index(drop=True)
)

timeseries_cohort.to_csv(OUTPUT_DIR / 'hourly_timeseries_60min.csv', index=False)
assessment_index.to_csv(OUTPUT_DIR / 'assessment_index_60min.csv', index=False)
cohort_final.to_csv(OUTPUT_DIR / 'cohort_final.csv', index=False)

print(f"Saved: hourly_timeseries_60min.csv ({len(timeseries_cohort):,} rows, {timeseries_cohort['stay_id'].nunique()} stays)")
print(f"Saved: assessment_index_60min.csv ({len(assessment_index):,} rows)")
print(f"Saved: cohort_final.csv ({len(cohort_final):,} stays)")
print(f"Saved: cohort_attrition.csv ({len(cohort_flow):,} rows)")


Saved: hourly_timeseries_60min.csv (74,230 rows, 573 stays)
Saved: assessment_index_60min.csv (5,320 rows)
Saved: cohort_final.csv (573 stays)
Saved: cohort_attrition.csv (4 rows)
